In [ ]:
import sys
import torch
import pickle
import os
from tqdm.notebook import tqdm

sys.path.insert(0, '..')
sys.path.insert(0, '../../')
sys.path.insert(0, '../../../')
sys.path.insert(0, '../../../../')
sys.path.insert(0, '../../../../../')
sys.path.insert(0, '../../../../../../')

from stochasticLSTM.model import StochasticLSTMWeytjens
from robustness.weytjens_evaluation import evaluate_with_predefined_prefixes



In [13]:
# Load model (two instances: stochastic for MC sampling, deterministic for point estimate)
file_path_model = '../../notebooks/training_variational_dropout/Helpdesk/Helpdesk_weytjens.pkl'
output_dir = '../../../../../evaluation_results/weytjens/helpdesk/last_event_attack_all/'

model = StochasticLSTMWeytjens.load(file_path_model, p_fix=0.05)
model_without_drop = StochasticLSTMWeytjens.load(file_path_model, p_fix=0)

# Load datasets
# Note: the Helpdesk dataset uses 'Activity' as the activity column name
file_path_original = '../../../../../encoded_data/weytjens/helpdesk/helpdesk_all_5_test.pkl'
file_path_redo_activity = '../../../../../encoded_data/weytjens/helpdesk/val_new.pkl'
file_path_redo_activity_pert = '../../../../../encoded_data/weytjens/helpdesk/val_new.pkl'

original_dataset = torch.load(file_path_original, weights_only=False)
redo_activity_dataset = torch.load(file_path_redo_activity, weights_only=False)
redo_activity_pert_dataset = torch.load(file_path_redo_activity_pert, weights_only=False)

print(f'Original dataset loaded: {len(original_dataset)} cases')
print(f'Clean pairs loaded: {len(redo_activity_dataset)} pairs')
print(f'Perturbed pairs loaded: {len(redo_activity_pert_dataset)} pairs')


Data set categories:  ([('Activity', 16, {'Assign seriousness': 1, 'Closed': 2, 'Create SW anomaly': 3, 'DUPLICATE': 4, 'EOS': 5, 'INVALID': 6, 'Insert ticket': 7, 'RESOLVED': 8, 'Require upgrade': 9, 'Resolve SW anomaly': 10, 'Resolve ticket': 11, 'Schedule intervention': 12, 'Take in charge ticket': 13, 'VERIFIED': 14, 'Wait': 15})], [('case_elapsed_time', 1, {})])
Model input features:  [['Activity'], ['case_elapsed_time']]


Embeddings:  ModuleList(
  (0): Embedding(16, 8)
)
Total embedding feature size:  8
Input feature size:  9
Cells hidden size:  10
Number of LSTM layer:  2
Dropout rate:  0.05


Output feature list of dicts (featue name, tensor index in dataset):  {'case_elapsed_time': 0}
Data set categories:  ([('Activity', 16, {'Assign seriousness': 1, 'Closed': 2, 'Create SW anomaly': 3, 'DUPLICATE': 4, 'EOS': 5, 'INVALID': 6, 'Insert ticket': 7, 'RESOLVED': 8, 'Require upgrade': 9, 'Resolve SW anomaly': 10, 'Resolve ticket': 11, 'Schedule intervention': 12, 'Take in charge t

In [14]:
def save_chunk(results, i, output_dir):
    """Save intermediate results to a numbered chunk file."""
    chunk_number = (i + 1)
    filename = os.path.join(output_dir, f'robustness_results_part_{chunk_number:03d}.pkl')
    with open(filename, 'wb') as f:
        pickle.dump(results, f)
    print(f'Saved {len(results)} results to {filename}')


In [15]:
os.makedirs(output_dir, exist_ok=True)

# Create evaluation generators for clean and perturbed prefix-suffix pairs.
# The Helpdesk dataset uses 'Activity' as the activity column name.
evaluate_with_predefined_prefixes_normal = evaluate_with_predefined_prefixes(
    model=model,
    model_without_drop=model_without_drop,
    dataset=original_dataset,
    predefined_pairs=redo_activity_dataset,
    device=torch.device('cpu'),
    samples_per_case=1,
    random_order=False,
    concept_name='Activity',
)

evaluate_with_predefined_prefixes_pert = evaluate_with_predefined_prefixes(
    model=model,
    model_without_drop=model_without_drop,
    dataset=original_dataset,
    predefined_pairs=redo_activity_pert_dataset,
    device=torch.device('cpu'),
    samples_per_case=1,
    random_order=False,
    concept_name='Activity',
)

print('Evaluation generators created')


Evaluation generators created


In [16]:
# Main evaluation loop
save_every = 50
results = {}

for i, ((case_name_orig, prefix_len_orig, prefix_orig, sampled_remaining_time_orig, suffix_orig, mean_remaining_time_orig),
        (case_name_pert, prefix_len_pert, prefix_pert, sampled_remaining_time_pert, suffix_pert, mean_remaining_time_pert)) in enumerate(
        tqdm(zip(evaluate_with_predefined_prefixes_normal, evaluate_with_predefined_prefixes_pert),
             desc='Evaluating robustness')):

    key = (case_name_orig, prefix_len_orig)

    #No prediction for suffix activities -> thus results are set to none to ensure metric calculation compatibilty
    mean_orig = []
    sampled_orig = []
    mean_pert = []
    sampled_pert = []

    # print(mean_remaining_time_orig)
    # print(sampled_remaining_time_orig)
    # print("--------------------------------")
    # assert(prefix_orig == prefix_pert)
    # assert(mean_remaining_time_orig[0]['case_elapsed_time'] == mean_remaining_time_pert[0]['case_elapsed_time'])

    results[key] = {
        'original': (prefix_orig, suffix_orig,mean_orig, sampled_orig, mean_remaining_time_orig[0]['case_elapsed_time'], sampled_remaining_time_orig),
        'perturbed': (prefix_pert, suffix_pert,mean_pert, sampled_pert, mean_remaining_time_pert[0]['case_elapsed_time'], sampled_remaining_time_pert)
    }

    if (i + 1) % save_every == 0:
        save_chunk(results, i, output_dir)
        results = {}

if len(results):
    save_chunk(results, i, output_dir)

print('Robustness evaluation completed!')


Evaluating robustness: 0it [00:00, ?it/s]

  0%|          | 0/1898 [00:00<?, ?it/s]

  0%|          | 0/1898 [00:00<?, ?it/s]

Saved 50 results to ../../../../../evaluation_results/weytjens/helpdesk/last_event_attack_all/robustness_results_part_050.pkl
Saved 50 results to ../../../../../evaluation_results/weytjens/helpdesk/last_event_attack_all/robustness_results_part_100.pkl
Saved 50 results to ../../../../../evaluation_results/weytjens/helpdesk/last_event_attack_all/robustness_results_part_150.pkl
Saved 50 results to ../../../../../evaluation_results/weytjens/helpdesk/last_event_attack_all/robustness_results_part_200.pkl
Saved 50 results to ../../../../../evaluation_results/weytjens/helpdesk/last_event_attack_all/robustness_results_part_250.pkl
Saved 50 results to ../../../../../evaluation_results/weytjens/helpdesk/last_event_attack_all/robustness_results_part_300.pkl
Saved 50 results to ../../../../../evaluation_results/weytjens/helpdesk/last_event_attack_all/robustness_results_part_350.pkl
Saved 50 results to ../../../../../evaluation_results/weytjens/helpdesk/last_event_attack_all/robustness_results_part_

In [17]:
# Load all saved chunks and combine them into a single results file
all_results = {}
chunk_files = sorted([f for f in os.listdir(output_dir) if f.startswith('robustness_results_part_')])

print(f'Found {len(chunk_files)} chunk files')

for chunk_file in chunk_files:
    chunk_path = os.path.join(output_dir, chunk_file)
    print(f'Loading {chunk_file}...')
    with open(chunk_path, 'rb') as f:
        chunk_results = pickle.load(f)
        all_results.update(chunk_results)
        print(f'  Added {len(chunk_results)} results from {chunk_file}')

if 'results' in locals() and len(results) > 0:
    print(f'Adding final {len(results)} results...')
    all_results.update(results)

print(f'\nTotal results loaded: {len(all_results)}')

combined_results_path = os.path.join(output_dir, 'robustness_results.pkl')
with open(combined_results_path, 'wb') as f:
    pickle.dump(all_results, f)

print(f'Combined results saved to {combined_results_path}')


Found 38 chunk files
Loading robustness_results_part_050.pkl...
  Added 50 results from robustness_results_part_050.pkl
Loading robustness_results_part_100.pkl...
  Added 50 results from robustness_results_part_100.pkl
Loading robustness_results_part_1000.pkl...
  Added 50 results from robustness_results_part_1000.pkl
Loading robustness_results_part_1050.pkl...
  Added 50 results from robustness_results_part_1050.pkl
Loading robustness_results_part_1100.pkl...
  Added 50 results from robustness_results_part_1100.pkl
Loading robustness_results_part_1150.pkl...
  Added 50 results from robustness_results_part_1150.pkl
Loading robustness_results_part_1200.pkl...
  Added 50 results from robustness_results_part_1200.pkl
Loading robustness_results_part_1250.pkl...
  Added 50 results from robustness_results_part_1250.pkl
Loading robustness_results_part_1300.pkl...
  Added 50 results from robustness_results_part_1300.pkl
Loading robustness_results_part_1350.pkl...
  Added 50 results from robust